In [14]:
!pip install sentence-transformers bertopic umap-learn pandas numpy matplotlib requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.1 MB/s eta 0:00:00


In this phase, we use the Crossref API to collect the DOI numbers of the papers belonging to a specific journal (via ISSN) within a defined year range. Then, we use the Semantic Scholar API to fetch the abstracts, titles, and metadata of these papers to save them locally.

In [ ]:
import requests
import json
import time

ISSN = "1351-3249"
START_YEAR = 2015
END_YEAR = 2025

# --- STEP 1A: Fetching DOIs ---
url = f"https://api.crossref.org/journals/{ISSN}/works"
all_dois = []
cursor = "*"

print("Collecting DOIs from Crossref API...")
while True:
    params = {"filter": f"from-pub-date:{START_YEAR},until-pub-date:{END_YEAR}", "select": "DOI,title,published,type", "rows": 100, "cursor": cursor}
    response = requests.get(url, params=params)
    data = response.json()
    items = data["message"]["items"]
    if not items: break
    for item in items:
        doi = item.get("DOI", "")
        title = item.get("title", [""])[0]
        year = item.get("published", {}).get("date-parts", [[None]])[0][0]
        all_dois.append({"doi": doi, "title": title, "year": year})
    cursor = data["message"].get("next-cursor")
    if not cursor: break
    time.sleep(1)

with open("dois.json", "w", encoding="utf-8") as f:
    json.dump(all_dois, f, ensure_ascii=False, indent=2)
print(f"Total: {len(all_dois)} DOIs saved.\n")

# --- STEP 1B: Fetching Abstracts with Real-time Saving ---
print("Fetching abstracts from Semantic Scholar...")
papers = []
not_found = 0

for i, item in enumerate(all_dois):
    doi = item.get("doi", "")
    if not doi: continue
    ss_url = f"https://api.semanticscholar.org/graph/v1/paper/{doi}"
    try:
        response = requests.get(ss_url, params={"fields": "title,abstract,year,authors"}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            abstract = data.get("abstract", "")
            if abstract:
                papers.append({
                    "doi": doi,
                    "title": data.get("title", item.get("title", "")),
                    "abstract": abstract,
                    "year": data.get("year", item.get("year")),
                    "authors": [a.get("name", "") for a in data.get("authors", [])]
                })
        else:
            not_found += 1
    except Exception as e:
        pass

    # HER 10 MAKALEDE BİR DİSKE KAYDET (Real-time Save)
    if (i + 1) % 10 == 0 or (i + 1) == len(all_dois):
        with open("papers.json", "w", encoding="utf-8") as f:
            json.dump({"papers": papers, "total": len(papers)}, f, ensure_ascii=False, indent=2)
        print(f"Progress: {i+1}/{len(all_dois)} — Saved to papers.json. Current valid papers: {len(papers)}")

    time.sleep(2) # Güvenli bekleme süresi

print(f"\nDone! Final total: {len(papers)} papers successfully saved.")

In this cell, the baseline model all-MiniLM-L6-v2 is utilized to project the journal's official "Aims & Scope" text and the retrieved paper abstracts into a dense vector space (embeddings). The resulting matrices are saved as .npy files.


In [9]:
# %% [code]
import json
import numpy as np
from sentence_transformers import SentenceTransformer

AIMS_AND_SCOPE = """
Natural Language Engineering is an open access journal which meets the needs
of professionals and researchers working in all areas of natural language
processing (NLP). Its aim is to bridge the gap between traditional
computational linguistics research and the implementation of practical
applications with potential real-world use. The journal publishes original
research articles on a broad range of methods and resources applied in NLP,
language processing tasks and NLP applications, including machine translation,
translation technology, sentiment analysis, information retrieval, question
answering, text summarisation, text simplification, and speech processing.
"""

MODEL_NAME = "all-MiniLM-L6-v2"
PAPERS_FILE = "papers.json"

print("Loading model: {}...".format(MODEL_NAME))
model = SentenceTransformer(MODEL_NAME)

print("Encoding Aims & Scope text...")
aims_embedding = model.encode(AIMS_AND_SCOPE.strip(), normalize_embeddings=True)
np.save("aims_embedding.npy", aims_embedding)
print("Aims & Scope embedding saved.")

print("Loading papers...")
with open(PAPERS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

papers = data["papers"]
abstracts = [p["abstract"] for p in papers]
print("Encoding {} abstracts...".format(len(abstracts)))

embeddings = model.encode(abstracts, normalize_embeddings=True, show_progress_bar=True)
np.save("paper_embeddings.npy", embeddings)
print("Done. Embeddings matrix shape: {}".format(embeddings.shape))

Loading model: all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding Aims & Scope text...
Aims & Scope embedding saved.
Loading papers...
Encoding 289 abstracts...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Done. Embeddings matrix shape: (289, 384)


Using the generated embeddings, semantic alignment scores (dot product/cosine similarity) are calculated between each paper and the journal's scope. Descriptive statistics are computed, and a threshold is set at the 5th percentile to identify potential outlier papers.


In [10]:
# %% [code]
import json
import numpy as np

PAPERS_FILE = "papers.json"

print("Loading embeddings...")
aims_embedding = np.load("aims_embedding.npy")
paper_embeddings = np.load("paper_embeddings.npy")

with open(PAPERS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

papers = data["papers"]
# Filter papers from 2015 onwards
papers = [p for p in papers if p.get("year") and int(p["year"]) >= 2015]

print("Computing alignment scores...")
scores = np.dot(paper_embeddings, aims_embedding)

for i, paper in enumerate(papers):
    paper["alignment_score"] = float(scores[i])

scores_array = np.array(scores)
print("\nAlignment Score Statistics:")
print("Mean:   {:.4f}".format(scores_array.mean()))
print("Std:    {:.4f}".format(scores_array.std()))
print("Min:    {:.4f}".format(scores_array.min()))
print("Max:    {:.4f}".format(scores_array.max()))
print("Median: {:.4f}".format(np.median(scores_array)))

threshold = np.percentile(scores_array, 5)
print("\nOutlier Threshold (5th percentile): {:.4f}".format(threshold))
outliers = [p for p in papers if p["alignment_score"] < threshold]
print("Number of Identified Outlier Papers: {}".format(len(outliers)))

with open("results.json", "w", encoding="utf-8") as f:
    json.dump({"papers": papers, "total": len(papers)}, f, ensure_ascii=False, indent=2)

print("\nResults successfully saved to 'results.json'.")

Loading embeddings...
Computing alignment scores...

Alignment Score Statistics:
Mean:   0.3585
Std:    0.1113
Min:    -0.0197
Max:    0.8417
Median: 0.3658

Outlier Threshold (5th percentile): 0.1517
Number of Identified Outlier Papers: 15

Results successfully saved to 'results.json'.


This cell benchmarks multiple Language Models (MiniLM-L6, MPNet-Base, and SPECTER). It compares their alignment score distributions, computes correlations between model outputs, and visualizes thematic drift across years.

In [11]:
# %% [code]
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

papers = data["papers"]
papers = [p for p in papers if p.get("year", 0) >= 2015]
abstracts = [p["abstract"] for p in papers]

print("Loaded paper count: {}".format(len(papers)))

models = {
    "MiniLM-L6"  : "all-MiniLM-L6-v2",
    "MPNet-Base" : "all-mpnet-base-v2",
    "SPECTER"    : "allenai-specter",
}

results = {}
for model_name, model_id in models.items():
    print("\nLoading model: {}...".format(model_name))
    model = SentenceTransformer(model_id)
    aims_vec = model.encode(AIMS_AND_SCOPE.strip()).reshape(1, -1)
    paper_vecs = model.encode(abstracts, show_progress_bar=True)
    scores = cosine_similarity(paper_vecs, aims_vec).flatten()
    results[model_name] = scores

df = pd.DataFrame(results)
df["year"] = [p["year"] for p in papers]
df["title"] = [p["title"] for p in papers]

# Plot 1: Score Distribution Histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for i, model_name in enumerate(models.keys()):
    axes[i].hist(df[model_name], bins=30, color="skyblue", edgecolor="black")
    axes[i].set_title("{}\nMean: {:.3f}".format(model_name, df[model_name].mean()))
    axes[i].set_xlabel("Alignment Score")
    axes[i].set_ylabel("Number of Papers" if i == 0 else "")
plt.suptitle("Alignment Score Distribution by Model", fontsize=13)
plt.tight_layout()
plt.savefig("model_comparison_histograms.png", dpi=200)
plt.close()

# Plot 2: Yearly Thematic Drift Analysis
plt.figure(figsize=(12, 5))
colors = ["steelblue", "darkorange", "green"]
for i, model_name in enumerate(models.keys()):
    yearly = df.groupby("year")[model_name].mean().reset_index().sort_values("year")
    plt.plot(yearly["year"], yearly[model_name], marker="o", label=model_name, color=colors[i])
plt.xlabel("Year")
plt.ylabel("Mean Alignment Score")
plt.title("Thematic Drift by Model (2015-2024)")
plt.legend()
plt.tight_layout()
plt.savefig("model_comparison_drift.png", dpi=200)
plt.close()

# Plot 3: Cross-Model Correlation Scatter Plots
model_names = list(models.keys())
pairs = [(model_names[0], model_names[1]), (model_names[0], model_names[2]), (model_names[1], model_names[2])]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (m1, m2) in enumerate(pairs):
    correlation = np.corrcoef(df[m1], df[m2])[0, 1]
    axes[i].scatter(df[m1], df[m2], alpha=0.3, s=5, color="steelblue")
    axes[i].set_xlabel("{} Score".format(m1))
    axes[i].set_ylabel("{} Score".format(m2))
    axes[i].set_title("{} vs {}\nCorrelation: {:.3f}".format(m1, m2, correlation))
plt.suptitle("Model Score Correlations", fontsize=13)
plt.tight_layout()
plt.savefig("model_comparison_correlation.png", dpi=200)
plt.close()

# Print Summary Table
print("\nMODEL COMPARISON SUMMARY:")
print("{:<15} {:>8} {:>8} {:>8} {:>8}".format('Model', 'Mean', 'Std', 'Min', 'Max'))
print("-" * 50)
for model_name in models.keys():
    s = df[model_name]
    print("{:<15} {:>8.4f} {:>8.4f} {:>8.4f} {:>8.4f}".format(model_name, s.mean(), s.std(), s.min(), s.max()))

df.to_csv("model_comparison_results.csv", index=False)

Loaded paper count: 288

Loading model: MiniLM-L6...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]


Loading model: MPNet-Base...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/462k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]


MODEL COMPARISON SUMMARY:
Model               Mean      Std      Min      Max
--------------------------------------------------
MiniLM-L6         0.3587   0.1116  -0.0197   0.8417
MPNet-Base        0.4461   0.1104  -0.0193   0.7922
SPECTER           0.7818   0.0572   0.5577   0.9588


This final cell fits a BERTopic model to extract core thematic clusters from the abstracts. It lists the most and least aligned topics based on semantic scores. Advanced plots are generated, including a Histogram, a Boxplot, a Percentile Trend chart, and a 2D UMAP projection showcasing the relative distance between papers and the target Aims & Scope.

In [15]:
# %% [code]
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

papers = data["papers"]
df = pd.DataFrame(papers)
df = df[df["year"] >= 2015].reset_index(drop=True)
abstracts = [p["abstract"] for p in papers]
embeddings = np.load("paper_embeddings.npy")[:len(df)]
aims_embedding = np.load("aims_embedding.npy")

# --- BERTopic Fitting ---
print("Fitting BERTopic Model...")
vectorizer = CountVectorizer(stop_words="english", min_df=2)
topic_model = BERTopic(language="english", vectorizer_model=vectorizer, calculate_probabilities=False, verbose=True)
topics, _ = topic_model.fit_transform(abstracts, embeddings)
df["topic_id"] = topics

# Topic Alignment Aggregation
topic_alignment = df.groupby("topic_id")["alignment_score"].mean().reset_index().sort_values("alignment_score")
topic_alignment = topic_alignment[topic_alignment["topic_id"] != -1]

print("\n5 Most Misaligned Topics:")
for _, row in topic_alignment.head(5).iterrows():
    words = [w[0] for w in topic_model.get_topic(int(row["topic_id"]))[:5]]
    print("  Topic {} [{:.3f}]: {}".format(int(row["topic_id"]), row["alignment_score"], ", ".join(words)))

print("\n5 Most Aligned Topics:")
for _, row in topic_alignment.tail(5).iterrows():
    words = [w[0] for w in topic_model.get_topic(int(row["topic_id"]))[:5]]
    print("  Topic {} [{:.3f}]: {}".format(int(row["topic_id"]), row["alignment_score"], ", ".join(words)))

# --- Advanced Visualizations ---
scores = df["alignment_score"].values

# 1. Histogram
plt.figure(figsize=(10, 5))
plt.hist(scores, bins=30, color="steelblue", edgecolor="black")
plt.xlabel("Alignment Score")
plt.ylabel("Number of Papers")
plt.title("Alignment Score Distribution")
plt.tight_layout()
plt.savefig("histogram.png", dpi=200)
plt.close()

# 2. Boxplot by Year
years = sorted(df["year"].unique())
data_by_year = [df[df["year"] == y]["alignment_score"].values for y in years]
plt.figure(figsize=(12, 5))
plt.boxplot(data_by_year, tick_labels=years)
plt.xlabel("Year")
plt.ylabel("Alignment Score")
plt.title("Alignment Score Distribution by Year")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("boxplot.png", dpi=200)
plt.close()

# 3. Percentile Trend (p10, median, p90)
p10 = df.groupby("year")["alignment_score"].quantile(0.10)
p50 = df.groupby("year")["alignment_score"].quantile(0.50)
p90 = df.groupby("year")["alignment_score"].quantile(0.90)
plt.figure(figsize=(10, 5))
plt.plot(p10.index, p10.values, label="p10", linestyle="--", color="red")
plt.plot(p50.index, p50.values, label="median", color="steelblue")
plt.plot(p90.index, p90.values, label="p90", linestyle="--", color="green")
plt.fill_between(p10.index, p10.values, p90.values, alpha=0.1, color="steelblue")
plt.xlabel("Year")
plt.ylabel("Alignment Score")
plt.title("Yearly Alignment Score Trend (p10 / median / p90)")
plt.legend()
plt.tight_layout()
plt.savefig("trend_p10_median_p90.png", dpi=200)
plt.close()

# 4. UMAP Dimensionality Reduction
try:
    from umap import UMAP
    reducer = UMAP(n_components=2, random_state=42)
    all_embeddings = np.vstack([embeddings, aims_embedding.reshape(1, -1)])
    reduced = reducer.fit_transform(all_embeddings)
    paper_reduced, aims_reduced = reduced[:-1], reduced[-1]

    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(paper_reduced[:, 0], paper_reduced[:, 1], c=df["alignment_score"].values, cmap="RdYlGn", alpha=0.7, s=20)
    plt.colorbar(scatter, label="Alignment Score")
    plt.scatter(aims_reduced[0], aims_reduced[1], c="black", s=200, marker="*", label="Aims & Scope", zorder=5)
    plt.title("UMAP Projection colored by Alignment Score")
    plt.legend()
    plt.tight_layout()
    plt.savefig("umap_alignment.png", dpi=200)
    plt.close()
except Exception as e:
    print("UMAP Visualization Error:", e)

df.to_csv("results_with_topics.csv", index=False)
print("\nAll pipeline benchmarks and plots generated successfully.")

2026-06-10 01:33:46,411 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic Model...


2026-06-10 01:33:56,735 - BERTopic - Dimensionality - Completed ✓
2026-06-10 01:33:56,737 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-10 01:33:56,751 - BERTopic - Cluster - Completed ✓
2026-06-10 01:33:56,755 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-10 01:33:56,818 - BERTopic - Representation - Completed ✓



5 Most Misaligned Topics:
  Topic 2 [0.314]: hate, speech, detection, content, data
  Topic 3 [0.327]: sentiment, analysis, classes, model, data
  Topic 1 [0.330]: abstract, world, ai, like, language
  Topic 4 [0.348]: semantic, lexical, similarity, word, embeddings
  Topic 0 [0.381]: language, models, languages, information, text

5 Most Aligned Topics:
  Topic 2 [0.314]: hate, speech, detection, content, data
  Topic 3 [0.327]: sentiment, analysis, classes, model, data
  Topic 1 [0.330]: abstract, world, ai, like, language
  Topic 4 [0.348]: semantic, lexical, similarity, word, embeddings
  Topic 0 [0.381]: language, models, languages, information, text

All pipeline benchmarks and plots generated successfully.
